# Рівень 3. Дообучення передобученої моделі (ResNet-18, ImageNet)

**Задача:** та сама, що на рівнях 1 і 2: за ASCII-рисунком визначити об'єкт (10 класів).
**Метод рівня 3:** transfer learning. Беремо згорткову мережу **ResNet-18**, яку вже навчили на наборі **ImageNet** (≈ 1.28 млн фотографій, 1000 класів), і **дообучаємо** (fine-tuning) її на наших рисунках.

**Чому саме так.** Висновки рівня 2 показали, що обмеження — не в задачі, а в кількості даних: мережа, навчена з нуля на 938 прикладах, швидко запам'ятовує train і перенавчається. Передобучена мережа вже вміє знаходити на зображеннях лінії, кути, дуги, текстури й частини об'єктів. Ці загальні знання не потрібно вчити з нуля з наших 938 рисунків, їх достатньо підлаштувати під ASCII-арт.

**Що залишається тим самим** (щоб порівняння було чесним):
- ті самі 1341 рисунок і **те саме розбиття** train / val / test з `splits.csv`;
- ті самі метрики (macro-F1 як основна, accuracy), той самий спосіб вимірювання часу і пам'яті;
- та сама функція `render()` (текст → зображення), що на рівнях 1 і 2.

**Залізо.** Як і рівень 2, фінальний запуск зроблено в Google Colab на GPU. Залізо кожного рівня вказано в порівняльній таблиці.

План:
1. Дані та розбиття
2. Що таке передобучена модель і fine-tuning
3. Підготовка зображень і аугментація
4. Три варіанти навчання
5. Вибір моделі та фінальна оцінка на test
6. Порівняння всіх трьох рівнів
7. Висновки

> **Як запустити.** У Google Colab: Runtime → Change runtime type → GPU, потім Runtime → Run all; повний прогін триває кілька хвилин. На CPU ноутбука він тривав би понад годину, тому для цього рівня GPU фактично обов'язковий. При першому запуску torchvision завантажує ваги ResNet-18 (≈ 45 MB).

## 1. Дані та розбиття

Якщо ноутбук запущено в Google Colab, спочатку завантажуємо репозиторій з датасетом і `splits.csv` з GitHub. Локально цей крок пропускається.

In [ ]:
import os
import subprocess
import sys

if 'google.colab' in sys.modules:
    if not os.path.exists('/content/ML'):
        subprocess.run(['git', 'clone', '-q', 'https://github.com/seibtoy/ML.git', '/content/ML'], check=True)
    os.chdir('/content/ML/level3')
print('Робоча папка:', os.getcwd())

In [ ]:
import copy
import json
import platform
import threading
import time

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import psutil
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from PIL import Image, ImageDraw, ImageFont
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, classification_report, f1_score

SEED = 42
DATASET_DIR = '../dataset'
SPLITS_PATH = '../splits.csv'
IMAGE_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 20

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


def cpu_name():
    if sys.platform == 'darwin':
        return subprocess.run(['sysctl', '-n', 'machdep.cpu.brand_string'], capture_output=True, text=True).stdout.strip()
    with open('/proc/cpuinfo') as f:
        return next(line.split(':', 1)[1].strip() for line in f if line.startswith('model name'))


hardware = f'GPU {torch.cuda.get_device_name(0)}' if device.type == 'cuda' else f'CPU {cpu_name()}'
print('Залізо:', hardware)
print('Python', platform.python_version())
for module in [torch, torchvision, np, pd, matplotlib, psutil]:
    print(module.__name__, module.__version__)
print('scikit-learn', __import__('sklearn').__version__, '| pillow', Image.__version__)

In [ ]:
splits = pd.read_csv(SPLITS_PATH)
texts = [open(os.path.join(DATASET_DIR, f), encoding='utf-8').read() for f in splits['file']]
classes = sorted(splits['label'].unique())
labels = splits['label'].map({c: i for i, c in enumerate(classes)}).to_numpy()

parts = {name: np.flatnonzero(splits['split'] == name) for name in ['train', 'val', 'test']}
y = {name: labels[idx] for name, idx in parts.items()}
{name: len(idx) for name, idx in parts.items()}

Клас `Measure` — той самий, що на рівні 2: час виконання блоку коду, пікова RAM процесу і пікова VRAM (пам'ять GPU). Перед зупинкою таймера чекаємо, поки GPU завершить усі обчислення (`torch.cuda.synchronize()`), інакше час був би заниженим.

In [ ]:
class Measure:
    def __enter__(self):
        self.process = psutil.Process()
        self.peak_rss = self.process.memory_info().rss
        self.running = True
        self.sampler = threading.Thread(target=self.sample, daemon=True)
        self.sampler.start()
        if device.type == 'cuda':
            torch.cuda.synchronize()
            torch.cuda.reset_peak_memory_stats()
        self.started = time.perf_counter()
        return self

    def sample(self):
        while self.running:
            self.peak_rss = max(self.peak_rss, self.process.memory_info().rss)
            time.sleep(0.005)

    def __exit__(self, *exc):
        if device.type == 'cuda':
            torch.cuda.synchronize()
        self.seconds = time.perf_counter() - self.started
        self.running = False
        self.sampler.join()
        self.peak_rss = max(self.peak_rss, self.process.memory_info().rss)
        self.peak_mb = self.peak_rss / 2**20
        self.peak_vram_mb = torch.cuda.max_memory_allocated() / 2**20 if device.type == 'cuda' else None

## 2. Що таке передобучена модель і fine-tuning

### ResNet-18

**ResNet-18** — згорткова нейромережа, яку запропонували дослідники Microsoft у 2015 році. Вона побудована з тих самих «цеглин», що й мережа рівня 2 (згортки, BatchNorm, ReLU, pooling, лінійний шар), але значно більша:
- **18 шарів з вагами** (17 згорткових і 1 лінійний) замість 8 згорткових у найкращої мережі рівня 2;
- **≈ 11.7 млн ваг** замість ≈ 1.7 млн, тобто приблизно в 7 разів більше;
- **залишкові з'єднання (residual connections)**: кожен блок не замінює свій вхід, а додає до нього поправку (`x + F(x)`). Сигнал і градієнт можуть проходити «в обхід» блоків, тому таку глибоку мережу вдається навчити.

### Передобучення на ImageNet

**ImageNet** — набір з ≈ 1.28 млн фотографій 1000 класів, серед яких багато порід котів і собак, птахи, літаки, човни, автомобілі, будівлі. На ньому ResNet-18 навчали кілька днів на кількох GPU. Готові ваги опубліковані в бібліотеці `torchvision` (`IMAGENET1K_V1`), і ми їх просто завантажуємо.

Що вміють шари передобученої мережі:
- **перші шари** знаходять краї, штрихи, кути (як HOG на рівні 1, але вивчене);
- **середні** — текстури та візерунки з цих штрихів;
- **глибокі** — частини об'єктів: очі, вуха, колеса, крила.

ASCII-рисунки — не фотографії, але на них намальовані ті самі об'єкти, і їхні контури складаються з тих самих штрихів і дуг. Тому знання з ImageNet мають бути корисними.

### Як пристосовуємо мережу до нашої задачі

Останній шар ResNet-18 (`fc`) рахує 1000 оцінок для класів ImageNet. Ми **замінюємо його новим лінійним шаром на 10 класів** (512 × 10 + 10 = 5 130 ваг) з випадковими початковими вагами. Далі є три варіанти навчання, і ми порівнюємо всі три:

1. **Заморожена мережа (feature extraction).** Ваги ResNet не змінюються, навчається лише новий останній шар. Мережа працює як готовий екстрактор ознак (ніби HOG з рівня 1, але вивчений на ImageNet), а останній шар — як логістична регресія над цими ознаками.
2. **Та сама ResNet-18 з випадковими вагами** (без передобучення). Це контрольний експеримент: він показує, що дає саме передобучення, а що — просто більша і сучасніша архітектура. Щоб порівняння було чесним, цю мережу навчаємо так само, як найкращі мережі рівня 2: довше і з label smoothing.
3. **Повний fine-tuning.** Починаємо з ваг ImageNet і дообучаємо **всі** 11.7 млн ваг. Learning rate невеликий, щоб підлаштувати знання мережі під ASCII-арт, а не стерти їх.

Процедура навчання (батчі по 32, AdamW, One-Cycle, вибір найкращої епохи за val macro-F1) — та сама, що на рівні 2.

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]


class ResNetClassifier(nn.Module):
    def __init__(self, pretrained, freeze_backbone):
        super().__init__()
        weights = torchvision.models.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None
        self.backbone = torchvision.models.resnet18(weights=weights)
        self.backbone.fc = nn.Linear(self.backbone.fc.in_features, len(classes))
        self.freeze_backbone = freeze_backbone
        if freeze_backbone:
            for name, parameter in self.backbone.named_parameters():
                parameter.requires_grad = name.startswith('fc.')
        self.register_buffer('mean', torch.tensor(IMAGENET_MEAN).view(1, 3, 1, 1))
        self.register_buffer('std', torch.tensor(IMAGENET_STD).view(1, 3, 1, 1))

    def train(self, mode=True):
        super().train(mode)
        if self.freeze_backbone:
            for module in self.backbone.modules():
                if isinstance(module, nn.BatchNorm2d):
                    module.eval()
        return self

    def forward(self, x):
        x = ((1 - x).repeat(1, 3, 1, 1) - self.mean) / self.std
        return self.backbone(x)


def count_parameters(model, trainable_only=False):
    return sum(p.numel() for p in model.parameters() if p.requires_grad or not trainable_only)

Кілька деталей у коді:
- **Вхід.** ResNet очікує кольорове зображення, нормалізоване середнім і стандартним відхиленням ImageNet. Наш рисунок сірий, тому ми повторюємо його канал тричі (R = G = B), перетворюємо на чорні символи на білому тлі (як папір) і нормалізуємо. Це робиться всередині `forward()`, тож мережа приймає ті самі зображення, що й на рівні 2.
- **Заморожування.** У замороженому варіанті `requires_grad = False` для всіх ваг, крім останнього шару, тому оптимізатор їх не змінює. Крім того, шари BatchNorm залишаються в режимі оцінки: інакше вони під час навчання оновлювали б свою статистику, і «заморожена» мережа все одно змінювалась би.

In [ ]:
def predict_logits(model, x):
    model.eval()
    with torch.no_grad():
        return torch.cat([model(x[i:i + 128].to(device)) for i in range(0, len(x), 128)])


def train(model, x_train, x_val, epochs, lr, augment=None, label_smoothing=0.0):
    model.to(device)
    x_train, x_val = x_train.to(device), x_val.to(device)
    generator = torch.Generator().manual_seed(SEED)
    counts = np.bincount(y['train'], minlength=len(classes))
    class_weights = torch.tensor(len(y['train']) / (len(classes) * counts), dtype=torch.float32, device=device)
    loss_fn = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=label_smoothing)
    optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=lr, total_steps=epochs, pct_start=0.15)
    y_train, y_val = torch.from_numpy(y['train']).to(device), torch.from_numpy(y['val']).to(device)

    history, best_f1, best_state = [], -1, None
    for epoch in range(epochs):
        model.train()
        order = torch.randperm(len(x_train), generator=generator).to(device)
        train_loss, train_pred = 0.0, torch.empty(len(x_train), dtype=torch.long, device=device)
        for start in range(0, len(order), BATCH_SIZE):
            batch = order[start:start + BATCH_SIZE]
            xb = augment(x_train[batch], generator) if augment else x_train[batch]
            logits = model(xb)
            loss = loss_fn(logits, y_train[batch])
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * len(batch)
            train_pred[batch] = logits.argmax(1)
        scheduler.step()

        val_logits = predict_logits(model, x_val)
        val_f1 = f1_score(y['val'], val_logits.argmax(1).cpu().numpy(), average='macro')
        history.append({
            'epoch': epoch + 1,
            'train loss': train_loss / len(x_train),
            'val loss': loss_fn(val_logits, y_val).item(),
            'train macro-F1': f1_score(y['train'], train_pred.cpu().numpy(), average='macro'),
            'val macro-F1': val_f1,
        })
        if val_f1 > best_f1:
            best_f1, best_state = val_f1, copy.deepcopy(model.state_dict())

    model.load_state_dict(best_state)
    return pd.DataFrame(history)


def plot_history(history, title):
    best = history.loc[history['val macro-F1'].idxmax()]
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    history.plot(x='epoch', y=['train loss', 'val loss'], ax=axes[0], title='Loss (менше — краще)')
    history.plot(x='epoch', y=['train macro-F1', 'val macro-F1'], ax=axes[1], title='Macro-F1 (більше — краще)')
    for ax in axes:
        ax.axvline(best['epoch'], color='gray', linestyle='--', linewidth=1)
    fig.suptitle(f"{title}: найкраща епоха {int(best['epoch'])}, val macro-F1 = {best['val macro-F1']:.3f}")
    plt.tight_layout()
    plt.show()

Функції `train()`, `predict_logits()` і `plot_history()` скопійовано з рівня 2 з однією зміною: оптимізатор отримує лише ті ваги, які навчаються (`requires_grad`), щоб у замороженому варіанті не обробляти 11 млн ваг, які не змінюються.

## 3. Підготовка зображень і аугментація

Рисунки рендеряться тією самою функцією `render()`, що на рівнях 1 і 2, але в розмірі **224×224** — це стандартний розмір входу ResNet, і на ньому в символів удвічі більше пікселів, ніж на 128×128 у рівні 2.

**Аугментація** (лише для train, на кожному батчі заново) — випадкові зміни, після яких об'єкт залишається тим самим:
- **віддзеркалення** по горизонталі з імовірністю 50%. Тут віддзеркалюються пікселі, тож `/` автоматично стає `\` (на рівні 2 для сітки символів це доводилося робити таблицею заміни);
- **масштаб** від 0.9 до 1.1;
- **зсув** до 10% розміру зображення по кожній осі.

Усе це одне афінне перетворення, яке PyTorch застосовує одразу до всього батчу (`affine_grid` + `grid_sample`), тому аугментація працює на GPU і майже не уповільнює навчання.

In [ ]:
FONT = ImageFont.truetype(os.path.join(matplotlib.get_data_path(), 'fonts/ttf/DejaVuSansMono.ttf'), 16)
CHAR_WIDTH = FONT.getlength('M')
LINE_HEIGHT = 19
MARGIN = 8


def render(text, size=IMAGE_SIZE):
    lines = text.rstrip('\n').split('\n')
    width = int(np.ceil(max(len(line) for line in lines) * CHAR_WIDTH))
    height = len(lines) * LINE_HEIGHT
    side = max(width, height) + 2 * MARGIN
    canvas = Image.new('L', (side, side), 255)
    draw = ImageDraw.Draw(canvas)
    left, top = (side - width) // 2, (side - height) // 2
    for i, line in enumerate(lines):
        draw.text((left, top + i * LINE_HEIGHT), line, fill=0, font=FONT)
    canvas = canvas.resize((size, size), Image.Resampling.LANCZOS)
    return 1 - np.asarray(canvas, dtype=np.float32) / 255


def to_images(texts):
    return torch.from_numpy(np.stack([render(t) for t in texts])).unsqueeze(1)


def augment_images(x, generator):
    n = len(x)
    flip = torch.where(torch.rand(n, generator=generator) < 0.5, -1.0, 1.0)
    scale = 0.9 + 0.2 * torch.rand(n, generator=generator)
    shift = 0.4 * (torch.rand(n, 2, generator=generator) - 0.5)
    theta = torch.zeros(n, 2, 3)
    theta[:, 0, 0] = flip / scale
    theta[:, 1, 1] = 1 / scale
    theta[:, :, 2] = shift
    grid = F.affine_grid(theta.to(x.device), list(x.shape), align_corners=False)
    return F.grid_sample(x, grid, padding_mode='zeros', align_corners=False)

In [ ]:
images, image_prep_cost = {}, {}
for name, idx in parts.items():
    with Measure() as m:
        images[name] = to_images([texts[i] for i in idx])
    image_prep_cost[name] = m
print({name: tuple(x.shape) for name, x in images.items()})

sample = images['train'][[i for i in range(len(y['train'])) if classes[y['train'][i]] == 'cat'][5]][None]
fig, axes = plt.subplots(1, 5, figsize=(16, 3.6))
axes[0].imshow(sample[0, 0], cmap='gray_r')
axes[0].set_title(f'Оригінал {IMAGE_SIZE}×{IMAGE_SIZE}')
augmented = augment_images(sample.repeat(4, 1, 1, 1), torch.Generator().manual_seed(1))
for ax, image in zip(axes[1:], augmented):
    ax.imshow(image[0], cmap='gray_r')
    ax.set_title('Аугментація')
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

## 4. Три варіанти навчання

Аугментація однакова для всіх трьох варіантів. Різняться початкові ваги, те, які ваги навчаються, кількість епох і learning rate:

| Варіант | Початкові ваги | Що навчається | Епох | Learning rate |
|---|---|---|---|---|
| Спроба 1 | ImageNet | лише останній шар | 20 | 3e-3 |
| Спроба 2 | випадкові | уся мережа | 80, label smoothing 0.1 | 1e-3 |
| Спроба 3 | ImageNet | уся мережа | 20 | 1e-3 |

Передобученим мережам вистачає 20 епох, бо вони починають не з нуля. Мережа з випадковими вагами за 20 епох не встигає навчитися, тому для чесного контрольного експерименту її навчаємо так само, як найкращі мережі рівня 2: довше (80 епох) і з label smoothing.

Для замороженої мережі learning rate більший, бо навчається лише один новий шар з випадковими вагами. Для всієї мережі він менший, щоб великі кроки не зруйнували корисні ваги ImageNet.

In [ ]:
def fit_attempt(title, pretrained, freeze_backbone, lr, epochs=EPOCHS, label_smoothing=0.0):
    torch.manual_seed(SEED)
    model = ResNetClassifier(pretrained, freeze_backbone)
    print(f'{title}: ваг у мережі {count_parameters(model):,}, з них навчаються {count_parameters(model, trainable_only=True):,}')
    with Measure() as fit_cost:
        history = train(model, images['train'], images['val'], epochs=epochs, lr=lr, augment=augment_images, label_smoothing=label_smoothing)
    print(f'Навчання: {fit_cost.seconds / 60:.1f} хв, пік RAM {fit_cost.peak_mb:.0f} MB, пік VRAM {fit_cost.peak_vram_mb} MB')
    plot_history(history, title)
    return model, history, fit_cost

### Спроба 1: заморожена ResNet-18, навчається лише останній шар

In [ ]:
frozen = fit_attempt('Спроба 1: заморожена ResNet-18', pretrained=True, freeze_backbone=True, lr=3e-3)

### Спроба 2: ResNet-18 з випадковими вагами (без передобучення)

In [ ]:
scratch = fit_attempt('Спроба 2: ResNet-18 з нуля', pretrained=False, freeze_backbone=False, lr=1e-3, epochs=80, label_smoothing=0.1)

### Спроба 3: повний fine-tuning ResNet-18

In [ ]:
finetuned = fit_attempt('Спроба 3: fine-tuning ResNet-18', pretrained=True, freeze_backbone=False, lr=1e-3)

## 5. Вибір моделі та фінальна оцінка на test

Як і раніше, модель обираємо **за validation**, а test використовуємо один раз.

In [ ]:
attempts = {
    'Спроба 1: заморожена ResNet-18': frozen,
    'Спроба 2: ResNet-18 з нуля': scratch,
    'Спроба 3: fine-tuning ResNet-18': finetuned,
}
summary = pd.DataFrame([
    {
        'спроба': name,
        'ваг, що навчаються': count_parameters(model, trainable_only=True),
        'епох': len(history),
        'найкраща епоха': int(history.loc[history['val macro-F1'].idxmax(), 'epoch']),
        'train macro-F1': history.loc[history['val macro-F1'].idxmax(), 'train macro-F1'],
        'val macro-F1': history['val macro-F1'].max(),
        'час навчання, хв': fit_cost.seconds / 60,
    }
    for name, (model, history, fit_cost) in attempts.items()
])
best_name = summary.loc[summary['val macro-F1'].idxmax(), 'спроба']
print('Обрано за validation:', best_name)
summary.round(3)

In [ ]:
final_model, final_history, final_fit_cost = attempts[best_name]

with Measure() as predict_cost:
    test_pred = predict_logits(final_model, images['test']).argmax(1).cpu().numpy()

train_seconds = image_prep_cost['train'].seconds + final_fit_cost.seconds
inference_seconds = image_prep_cost['test'].seconds + predict_cost.seconds

test_accuracy = accuracy_score(y['test'], test_pred)
test_macro_f1 = f1_score(y['test'], test_pred, average='macro')

print(f'Модель: {best_name}')
print(f'Test accuracy: {test_accuracy:.3f}')
print(f'Test macro-F1: {test_macro_f1:.3f}')
print()
print(classification_report(y['test'], test_pred, target_names=classes, digits=3))

**Наскільки точна ця оцінка.** Test — лише 202 рисунки, тому macro-F1 на ньому сам по собі має випадкову похибку. Оцінимо її **бутстрепом**: 2000 разів випадково вибираємо 202 рисунки з test з поверненням (деякі потрапляють двічі, деякі не потрапляють) і рахуємо macro-F1 на кожній такій вибірці. Проміжок, у який потрапляють середні 95% значень, — це 95% довірчий інтервал. Якщо його нижня межа вища за результат рівня 2, перевага рівня 3 не пояснюється випадковістю вибору test.

In [ ]:
rng = np.random.default_rng(SEED)
bootstrap = []
for _ in range(2000):
    sample_idx = rng.integers(0, len(test_pred), len(test_pred))
    bootstrap.append(f1_score(y['test'][sample_idx], test_pred[sample_idx], average='macro'))
ci_low, ci_high = np.percentile(bootstrap, [2.5, 97.5])
print(f'Test macro-F1 = {test_macro_f1:.3f}, 95% довірчий інтервал: [{ci_low:.3f}, {ci_high:.3f}]')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))
ConfusionMatrixDisplay.from_predictions(
    [classes[i] for i in y['test']], [classes[i] for i in test_pred],
    labels=classes, xticks_rotation=45, colorbar=False, ax=ax,
)
ax.set_title('Матриця помилок на test')
ax.set_xlabel('передбачений клас')
ax.set_ylabel('справжній клас')
plt.show()

In [ ]:
wrong = np.flatnonzero(test_pred != y['test'])
fig, axes = plt.subplots(2, 5, figsize=(14, 6))
for ax, i in zip(axes.flat, wrong[:10]):
    ax.imshow(images['test'][i, 0], cmap='gray_r')
    ax.set_title(f"справжній: {classes[y['test'][i]]}\nмодель: {classes[test_pred[i]]}", fontsize=10)
    ax.axis('off')
plt.suptitle('Приклади помилок на test')
plt.tight_layout()
plt.show()

**Час і пам'ять** рахуються так само, як на рівнях 1 і 2:
- **навчання** = рендер train + навчання мережі (разом з оцінкою на validation після кожної епохи). Завантаження готових ваг ImageNet у час навчання не входить: це разове скачування файлу, а не обчислення;
- **інференс** = рендер test + прохід мережі по test;
- **RAM** — пік процесу; **VRAM** — пік пам'яті GPU.

In [ ]:
final_costs = [*image_prep_cost.values(), final_fit_cost, predict_cost]
vram = [m.peak_vram_mb for m in final_costs if m.peak_vram_mb is not None]

results = {
    'level': 3,
    'method': f'Transfer learning, ResNet-18 ImageNet ({best_name.split(": ")[1]})',
    'hardware': hardware,
    'test_accuracy': round(test_accuracy, 4),
    'test_macro_f1': round(test_macro_f1, 4),
    'train_seconds': round(train_seconds, 3),
    'inference_seconds': round(inference_seconds, 3),
    'inference_ms_per_sample': round(1000 * inference_seconds / len(parts['test']), 3),
    'peak_ram_mb': round(max(m.peak_mb for m in final_costs)),
    'peak_vram_mb': round(max(vram)) if vram else None,
}
with open('results.json', 'w') as f:
    json.dump(results, f, ensure_ascii=False, indent=2)
print(json.dumps(results, ensure_ascii=False, indent=2))

pd.DataFrame([
    ['Залізо', hardware],
    ['Метрика на test: accuracy', f'{test_accuracy:.3f}'],
    ['Метрика на test: macro-F1', f'{test_macro_f1:.3f} (95% ДІ {ci_low:.3f}–{ci_high:.3f})'],
    ['Час навчання: рендер train', f"{image_prep_cost['train'].seconds:.2f} с"],
    ['Час навчання: навчання мережі', f'{final_fit_cost.seconds:.1f} с ({final_fit_cost.seconds / 60:.1f} хв)'],
    ['Час навчання: разом', f'{train_seconds:.1f} с'],
    ['Час інференсу: рендер test', f"{image_prep_cost['test'].seconds:.3f} с"],
    ['Час інференсу: прохід мережі', f'{predict_cost.seconds:.3f} с'],
    ['Час інференсу: разом', f'{inference_seconds:.3f} с ({results["inference_ms_per_sample"]:.2f} мс на рисунок)'],
    ['Пікова RAM процесу', f"{results['peak_ram_mb']} MB"],
    ['Пікова VRAM', f"{results['peak_vram_mb']} MB" if vram else 'не використовується (CPU)'],
], columns=['Показник', 'Значення'])

## 6. Порівняння всіх трьох рівнів

Усі рівні оцінені на тих самих 202 рисунках test. Рівень 1 рахувався на CPU ноутбука, рівні 2 і 3 — на GPU в Colab: так кожен метод використовують на практиці, і потреба в GPU — теж частина ціни методу.

In [ ]:
levels = []
for path in ['../level1/results.json', '../level2/results.json']:
    with open(path) as f:
        levels.append(json.load(f))
levels[0].setdefault('hardware', 'CPU Intel(R) Core(TM) i7-8850H CPU @ 2.60GHz')

comparison = pd.DataFrame([*levels, results]).set_index('level')
comparison = comparison[['method', 'hardware', 'test_accuracy', 'test_macro_f1', 'train_seconds', 'inference_ms_per_sample', 'peak_ram_mb', 'peak_vram_mb']]
comparison.columns = ['метод', 'залізо', 'test accuracy', 'test macro-F1', 'навчання, с', 'інференс, мс/рисунок', 'пік RAM, MB', 'пік VRAM, MB']
comparison

## 7. Висновки

_(буде заповнено після запуску в Colab)_